## Module overview

This notebook imports the core components of the project from the `Python/` package.

- `Python.framework`  
  Contains the main experiment pipeline, including stagewise annealing training, checkpoint evaluation and selection, final checkpoint summarization, and the high-level wrapper `simflow_stagewise`.

- `Python.model`  
  Defines the model architecture, including the flow-based latent transport maps, the relaxed spike-and-slab generative model, the variational base distribution, and the `build_flow_vi` constructor.

- `Python.simfun`  
  Provides simulation utilities for generating synthetic regression datasets with sparse ground-truth coefficients.

- `Python.config`  
  Stores configuration dataclasses used throughout the pipeline, including annealing, data splitting, and saving options.

The imports below allow the notebook to call the full training-and-evaluation pipeline while keeping the implementation organized across separate source files.

In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import Python.framework as fw
import Python.model as md
import Python.simfun as sf
import Python.config as cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Experiment setup and execution

This block defines the full experimental configuration and runs one complete stagewise annealing experiment.

- `schedule_cfg` specifies the optimization strategy, including the temperature schedule, warm-up length, number of annealing stages, stagewise epoch limits, learning-rate decay, evaluation frequency, posterior diagnostic sample sizes, and the hard-support threshold.
- `split_cfg` defines the train/validation/test split used for checkpoint evaluation and final reporting.
- `save_cfg` controls whether outputs such as plots, histories, checkpoint manifests, and summary tables are written to disk.
- `fw.simflow_stagewise(...)` executes the full pipeline: data generation, model construction, stagewise annealing, checkpoint selection, final posterior summarization, and optional result saving.

This block is the main entry point for testing the current framework under a chosen experimental configuration.

In [21]:
schedule_cfg = cfg.StagewiseAnnealConfig(
    tau_start=1,
    tau_end=0.1,
    warmup_epochs=300,
    n_anneal_stages=10,
    min_stage_epochs=100,
    max_stage_epochs=300,
    base_lr=5e-5,
    stage_lr_decay=0.7,
    eval_every=25,
    print_every=100,
    diag_R_train=256,
    diag_R_final=4000,
    support_threshold=0.1,
)

split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)

save_cfg = cfg.SaveConfig(
    output_dir="./plot/run_seed123",
    save_plots=True,
    save_history_csv=True,
    save_checkpoint_manifest=True,
    save_final_json=True,
    save_predictions_csv=True,
    save_support_sets_json=True,
)

out1 = fw.simflow_stagewise(
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    simfun=sf.simfun1,
    seed=123,
    n=180,
    p=100,
    snr=3.0,
    true_prop=0.1,
    K_q=8,
    K_g=8,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'n': 180, 'p': 100, 'n_active': 10, 'sigma2': 3.2589944203694663, 'sigma': 1.8052685175257077, 'active_idx': array([14, 17, 20, 32, 35, 43, 45, 62, 66, 95], dtype=int64), 'snr': 3.0}
[stage 00 | epoch 0001] tau=1.0000 lr=5.00e-05 loss=767.312805 logg=5.8049 dlogg=+0.0000 S=100 churn=0.000 ok*
[stage 00 | epoch 0100] tau=1.0000 lr=5.00e-05 loss=562.496582 logg=4.9479 dlogg=-0.5800 S= 82 churn=0.180 ok
[stage 00 | epoch 0200] tau=1.0000 lr=5.00e-05 loss=283.563751 logg=5.3912 dlogg=+0.2965 S= 15 churn=0.756 ok*
[stage 00 | epoch 0300] tau=1.0000 lr=5.00e-05 loss=216.562057 logg=5.3417 dlogg=-1.0967 S=  4 churn=0.429 ok
[stage 01 | epoch 0301] tau=0.7943 lr=5.00e-05 loss=216.637222 logg=5.4074 dlogg=+0.0657 S=  5 churn=0.200 ok*
[stage 01 | epoch 0400] tau=0.7943 lr=5.00e-05 loss=198.477814 logg=5.4120 dlogg=+0.0748 S=  4 churn=0.200 ok
[stage 01 | epoch 0500] tau=0.7943 lr=5.00e-05 loss=191.322464 logg=5.4132 dlogg=+0.4060 S=  4 churn=0.000 ok
[stage 01 | epo

In [1]:
import numpy as np
import pandas as pd

from meanfield_benchmark_module import (
    SplitConfig,
    MFSpikeSlabConfig,
    MFARDConfig,
    MFBayesLassoConfig,
    run_setting_grid,
)


def simfun_demo(seed: int, n: int, p: int, snr: float, true_prop: float):
    rng = np.random.default_rng(seed)
    s = max(1, int(round(p * true_prop)))
    active_idx = np.sort(rng.choice(p, size=s, replace=False))

    X = rng.normal(size=(n, p))
    beta_true = np.zeros(p)
    beta_true[active_idx] = rng.normal(size=s)

    signal = X @ beta_true
    sigma2 = np.var(signal, ddof=0) / snr
    y = signal + rng.normal(scale=np.sqrt(max(sigma2, 1e-8)), size=n)

    return {
        "X": X,
        "y": y,
        "beta_true": beta_true,
        "active_idx": active_idx,
        "sigma2": sigma2,
        "snr": snr,
        "n_active": s,
    }


split_cfg = SplitConfig(train_frac=0.6, val_frac=0.2, test_frac=0.2, seed=123)

method_cfgs = {
    "mf_spike_slab": MFSpikeSlabConfig(pi=0.10, support_threshold=0.5, beta_eps=0.10, max_iter=500),
    "mf_ard": MFARDConfig(support_threshold=0.5, beta_eps=0.10, max_iter=500),
    "mf_bayes_lasso": MFBayesLassoConfig(lasso_lambda=1.0, support_threshold=0.5, beta_eps=0.10, max_iter=500),
}

setting = {
    "n": 180,
    "p": 100,
    "snr": 3.0,
    "true_prop": 0.10,
}

results, table = run_setting_grid(
    simfun=simfun_demo,
    setting=setting,
    seeds=[101, 102, 103, 104, 105],
    methods=["mf_spike_slab", "mf_ard", "mf_bayes_lasso"],
    split_cfg=split_cfg,
    method_cfgs=method_cfgs,
)

print(table)

summary = (
    table.groupby("method")[["precision", "recall", "f1", "test_mse", "runtime_sec"]]
    .agg(["mean", "std"])
)

print("\n===== summary over seeds =====")
print(summary)

            method  seed  runtime_sec  converged  n_iter  support_size  \
0    mf_spike_slab   101     0.010469       True      12           6.0   
1           mf_ard   101     0.016990       True      45          79.0   
2   mf_bayes_lasso   101     0.097637       True     283          81.0   
3    mf_spike_slab   102     0.022919       True      37           7.0   
4           mf_ard   102     0.017872       True      54          40.0   
5   mf_bayes_lasso   102     0.094230       True     270          73.0   
6    mf_spike_slab   103     0.007061       True      11           6.0   
7           mf_ard   103     0.023437       True      67          51.0   
8   mf_bayes_lasso   103     0.141887       True     391          74.0   
9    mf_spike_slab   104     0.010248       True      15           6.0   
10          mf_ard   104     0.031846       True      91          51.0   
11  mf_bayes_lasso   104     0.059748       True     174          65.0   
12   mf_spike_slab   105     0.012110 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from meanfield_benchmark_module import (
    SplitConfig,
    SaveConfig,
    MFSpikeSlabConfig,
    MFARDConfig,
    MFBayesLassoConfig,
    run_full_benchmark,
    save_result_artifacts,
    save_benchmark_table,
    plot_runtime_vs_f1,
    plot_test_mse_vs_support_size,
    plot_precision_recall,
    plot_support_score_rank,
)


def simfun_adapter(seed: int, n: int, p: int, snr: float, true_prop: float):
    rng = np.random.default_rng(seed)

    s = max(1, int(round(p * true_prop)))
    active_idx = np.sort(rng.choice(p, size=s, replace=False))

    X = rng.normal(size=(n, p))

    beta_true = np.zeros(p)
    beta_true[active_idx] = rng.normal(size=s)

    signal = X @ beta_true
    sigma2 = np.var(signal, ddof=0) / snr
    y = signal + rng.normal(scale=np.sqrt(max(sigma2, 1e-8)), size=n)

    return {
        "X": X,
        "y": y,
        "beta_true": beta_true,
        "active_idx": active_idx,
        "sigma2": sigma2,
        "snr": snr,
        "n_active": s,
    }


PROJECT_NAME = "meanfield_only"
OUTPUT_DIR = Path("./benchmark") / PROJECT_NAME

SEEDS = [101, 102, 103, 104, 105]

METHODS = [
    "mf_spike_slab",
    "mf_ard",
    "mf_bayes_lasso",
]

setting_grid = [
    {
        "n": 180,
        "p": 100,
        "snr": 3.0,
        "true_prop": 0.10,
    },
]


split_cfg = SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)


save_cfg = SaveConfig(
    output_dir=str(OUTPUT_DIR),
    save_history_csv=True,
    save_final_json=True,
    save_predictions_csv=True,
    save_var_table_csv=True,
    save_benchmark_csv=True,
    save_plots=True,
)


method_cfgs = {
    "mf_spike_slab": MFSpikeSlabConfig(
        pi=0.10,
        slab_var=1.0,
        a_sigma=1.0,
        b_sigma=1.0,
        update_sigma2=True,
        min_sigma2=1e-8,
        support_threshold=0.5,
        beta_eps=0.10,
        standardize_x=True,
        center_y=True,
        max_iter=500,
        tol=1e-5,
        verbose=False,
    ),
    "mf_ard": MFARDConfig(
        a0=1e-2,
        b0=1e-2,
        c0=1e-2,
        d0=1e-2,
        min_sigma2=1e-8,
        support_threshold=0.5,
        beta_eps=0.10,
        standardize_x=True,
        center_y=True,
        max_iter=500,
        tol=1e-5,
        verbose=False,
    ),
    "mf_bayes_lasso": MFBayesLassoConfig(
        lasso_lambda=1.0,
        c0=1e-2,
        d0=1e-2,
        min_sigma2=1e-8,
        support_threshold=0.5,
        beta_eps=0.10,
        standardize_x=True,
        center_y=True,
        max_iter=500,
        tol=1e-5,
        verbose=False,
    ),
}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results, table = run_full_benchmark(
    simfun=simfun_adapter,
    setting_grid=setting_grid,
    seeds=SEEDS,
    methods=METHODS,
    split_cfg=split_cfg,
    method_cfgs=method_cfgs,
)

print("\n===== Raw benchmark table =====")
print(table.to_string(index=False))


summary_cols = [
    "precision",
    "recall",
    "f1",
    "support_size",
    "test_mse",
    "test_r2",
    "runtime_sec",
]

summary = (
    table.groupby(["n", "p", "snr", "true_prop", "method"])[summary_cols]
    .agg(["mean", "std"])
    .reset_index()
)

summary_path = OUTPUT_DIR / "benchmark_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n===== Summary over seeds =====")
print(summary.to_string(index=False))

print("\nSummary saved to:")
print(summary_path.resolve())


for res in results:
    save_result_artifacts(res, save_cfg)

save_benchmark_table(table, save_cfg)


plot_runtime_vs_f1(
    table,
    output_path=str(OUTPUT_DIR / "runtime_vs_f1.png"),
)

plot_test_mse_vs_support_size(
    table,
    output_path=str(OUTPUT_DIR / "test_mse_vs_support_size.png"),
)

plot_precision_recall(
    table,
    output_path=str(OUTPUT_DIR / "precision_recall.png"),
)

for res in results:
    method = res["method"]
    seed = res["seed"]

    n = res["sim_info"].get("n", "NA")
    p = res["sim_info"].get("p", "NA")
    snr = res["sim_info"].get("snr", "NA")
    true_prop = res["sim_info"].get("true_prop", "NA")

    fig_name = (
        f"support_score_rank_"
        f"{method}_"
        f"n{n}_p{p}_snr{snr}_prop{true_prop}_seed{seed}.png"
    )

    plot_support_score_rank(
        res,
        output_path=str(OUTPUT_DIR / fig_name),
    )

plt.close("all")


print("\nArtifacts saved to:")
print(OUTPUT_DIR.resolve())

print("\nFiles:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)


===== Raw benchmark table =====
        method  seed  runtime_sec  converged  n_iter  support_size  precision  recall       f1      fdr  tp   fp  fn  train_mse   val_mse  test_mse  train_r2    val_r2   test_r2   val_nll  test_nll   n   p  snr  true_prop  n_active
 mf_spike_slab   101     0.008098       True      12           6.0   1.000000     0.6 0.750000 0.000000 6.0  0.0 4.0   4.431449  5.780758  6.699626  0.771509  0.726578  0.550271  2.303243  2.396810 180 100  3.0        0.1        10
        mf_ard   101     0.014919       True      45          79.0   0.113924     0.9 0.202247 0.886076 9.0 70.0 1.0   2.321150  8.589322  9.597166  0.880318  0.593737  0.355766  2.604051  2.710638 180 100  3.0        0.1        10
mf_bayes_lasso   101     0.099197       True     283          81.0   0.111111     0.9 0.197802 0.888889 9.0 72.0 1.0   0.728864 30.864889 26.891467  0.962419 -0.459866 -0.805158 14.822802 13.039839 180 100  3.0        0.1        10
 mf_spike_slab   102     0.024447      

config.py
    StagewiseAnnealConfig
    SplitConfig
    SaveConfig
    MeanFieldBenchmarkConfig
    MFSpikeSlabConfig / MFARDConfig / MFBayesLassoConfig 

utils.py
    Array = np.ndarray
    to_numpy
    safe_float
    EMA
    jaccard_distance
    jaccard_similarity
    rolling_stage_taus
    set_optimizer_lr
    make_optimizer
    save_model_state
    load_model_state
    ensure_dir
    split_data_tensors
    make_splits / make_split_indices
    standardize_design
    center_response
    recover_original_scale
    predict_linear
    NumpyJSONEncoder

metric.py
    sample_posterior_latents
    hard_support_from_draws
    posterior_predictions_from_hard_samples
    predictive_metrics
    posterior_predictive_metrics_from_hard_samples
    gaussian_predictive_metrics
    selection_metrics_from_support
    support_metrics
    normal_cdf
    prob_abs_gt_eps

simfun.py
    simfun1
    simfun2
    extract_sim_arrays

diagplot.py
    flow training plot
    boundary plot
    support overlap plot
    benchmark comparison plot

artifact.py
    save_run_artifacts
    save_flow_run_artifacts alias
    save_result_artifacts
    save_meanfield_result_artifacts alias
    save_benchmark_table

mf_spike_slab_model.py
    MFSpikeSlabConfig
    _fit_mf_spike_slab
    run_mf_spike_slab

mf_ard_model.py
    MFARDConfig
    _fit_mf_ard
    run_mf_ard

mf_bayes_lasso_model.py
    MFBayesLassoConfig
    _fit_mf_bayes_lasso
    run_mf_bayes_lasso

meanfield_benchmark_core.py
    top_var_table
    benchmark_row_from_result
    _finalize_linear_result
    adapt_existing_spike_slab_output
    run_mcmc_placeholder
    get_method_registry
    run_baseline_method
    run_one_setting_one_seed
    run_setting_grid
    run_full_benchmark

meanfield_benchmark_module.py
     re-export / compatibility layer

In [7]:
from pathlib import Path
from typing import Callable, Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import meanfield_benchmark_module as mf


def test_three_meanfield_models(
    *,
    simfun: Optional[Callable] = None,
    seeds: Sequence[int] = (101, 102, 103, 104, 105),
    n: int = 180,
    p: int = 100,
    snr: float = 3.0,
    true_prop: float = 0.10,
    output_dir: str = "./benchmark/test_three_meanfield",
    save_outputs: bool = True,
):
    """
    Smoke / benchmark test for three mean-field baselines:
      1. mf_spike_slab
      2. mf_ard
      3. mf_bayes_lasso

    If simfun is None, use an internal synthetic Gaussian linear simulation.
    If simfun is provided, it must accept:
        seed, n, p, snr, true_prop
    and return a payload readable by simfun.extract_sim_arrays.
    """

    # ------------------------------------------------------------------
    # 1. Default simulation function
    # ------------------------------------------------------------------
    def default_simfun(seed: int, n: int, p: int, snr: float, true_prop: float):
        rng = np.random.default_rng(seed)

        n_active = max(1, int(round(p * true_prop)))
        active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

        X = rng.normal(size=(n, p))

        beta_true = np.zeros(p)
        beta_true[active_idx] = rng.normal(loc=0.0, scale=1.0, size=n_active)

        signal = X @ beta_true
        signal_var = float(np.var(signal, ddof=0))
        sigma2 = signal_var / max(snr, 1e-8)

        y = signal + rng.normal(scale=np.sqrt(max(sigma2, 1e-8)), size=n)

        return {
            "X": X,
            "y": y,
            "beta_true": beta_true,
            "active_idx": active_idx,
            "sigma2": sigma2,
            "snr": snr,
            "n_active": n_active,
        }

    if simfun is None:
        simfun = default_simfun

    # ------------------------------------------------------------------
    # 2. Shared configs
    # ------------------------------------------------------------------
    split_cfg = mf.SplitConfig(
        train_frac=0.60,
        val_frac=0.20,
        test_frac=0.20,
        seed=123,
    )

    save_cfg = mf.SaveConfig(
        output_dir=output_dir,
        save_plots=True,
        save_history_csv=True,
        save_final_json=True,
        save_predictions_csv=True,
        save_var_table_csv=True,
        save_benchmark_csv=True,
    )

    methods = [
        "mf_spike_slab",
        "mf_ard",
        "mf_bayes_lasso",
    ]

    method_cfgs = {
        "mf_spike_slab": mf.MFSpikeSlabConfig(
            pi=0.10,
            slab_var=1.0,
            support_threshold=0.5,
            beta_eps=0.10,
            standardize_x=True,
            center_y=True,
            max_iter=500,
            tol=1e-5,
            verbose=False,
        ),
        "mf_ard": mf.MFARDConfig(
            support_threshold=0.5,
            beta_eps=0.10,
            standardize_x=True,
            center_y=True,
            max_iter=500,
            tol=1e-5,
            verbose=False,
        ),
        "mf_bayes_lasso": mf.MFBayesLassoConfig(
            lasso_lambda=1.0,
            support_threshold=0.5,
            beta_eps=0.10,
            standardize_x=True,
            center_y=True,
            max_iter=500,
            tol=1e-5,
            verbose=False,
        ),
    }

    setting_grid = [
        {
            "n": n,
            "p": p,
            "snr": snr,
            "true_prop": true_prop,
        }
    ]

    # ------------------------------------------------------------------
    # 3. Run benchmark
    # ------------------------------------------------------------------
    results, table = mf.run_full_benchmark(
        simfun=simfun,
        setting_grid=setting_grid,
        seeds=list(seeds),
        methods=methods,
        split_cfg=split_cfg,
        method_cfgs=method_cfgs,
    )

    # ------------------------------------------------------------------
    # 4. Print raw table
    # ------------------------------------------------------------------
    display_cols = [
        "method",
        "seed",
        "support_size",
        "precision",
        "recall",
        "f1",
        "fdr",
        "test_mse",
        "test_r2",
        "runtime_sec",
    ]

    existing_cols = [c for c in display_cols if c in table.columns]

    print("\n===== Raw benchmark table =====")
    print(table[existing_cols].to_string(index=False))

    # ------------------------------------------------------------------
    # 5. Summary over seeds
    # ------------------------------------------------------------------
    summary_cols = [
        "support_size",
        "precision",
        "recall",
        "f1",
        "fdr",
        "test_mse",
        "test_r2",
        "runtime_sec",
    ]

    summary_cols = [c for c in summary_cols if c in table.columns]

    summary = (
        table.groupby("method")[summary_cols]
        .agg(["mean", "std"])
        .reset_index()
    )

    print("\n===== Summary over seeds =====")
    print(summary.to_string(index=False))

    # ------------------------------------------------------------------
    # 6. Save artifacts and plots
    # ------------------------------------------------------------------
    if save_outputs:
        outdir = Path(output_dir)
        outdir.mkdir(parents=True, exist_ok=True)

        for res in results:
            mf.save_result_artifacts(res, save_cfg)

        mf.save_benchmark_table(table, save_cfg)

        summary.to_csv(outdir / "benchmark_summary.csv", index=False)

        mf.plot_runtime_vs_f1(
            table,
            output_path=str(outdir / "runtime_vs_f1.png"),
        )

        mf.plot_test_mse_vs_support_size(
            table,
            output_path=str(outdir / "test_mse_vs_support_size.png"),
        )

        mf.plot_precision_recall(
            table,
            output_path=str(outdir / "precision_recall.png"),
        )

        for res in results:
            method = res.get("method", "unknown")
            seed = res.get("seed", "na")

            mf.plot_support_score_rank(
                res,
                output_path=str(outdir / f"support_score_rank_{method}_seed{seed}.png"),
            )

        plt.close("all")

        print("\nArtifacts saved to:")
        print(outdir.resolve())

    return {
        "results": results,
        "table": table,
        "summary": summary,
        "split_cfg": split_cfg,
        "method_cfgs": method_cfgs,
        "save_cfg": save_cfg,
    }